In [13]:
import json
import pandas as pd

# Path to your JSON file
file_path = "/home/pushpendras0026/dialog2flow/data/final_annotated_dataset_with_satisfaction.json"

# Load JSON
with open(file_path, "r") as f:
    data = json.load(f)

print(f"Number of records: {len(data)}")
print("Keys in first record:", data[0].keys())

# Convert to DataFrame for easier exploration
df = pd.DataFrame(data)
print("\nDataFrame shape:", df.shape)


Number of records: 18
Keys in first record: dict_keys(['transcript_id', 'time_of_interaction', 'domain', 'intent', 'reason_for_call', 'conversation', 'satisfaction_score', 'satisfaction_label'])

DataFrame shape: (18, 8)


In [8]:
domain_intent_counts = (
    df.groupby('domain')['intent']
      .value_counts()
      .reset_index(name='count')
)

In [12]:
annotated_path = "/home/pushpendras0026/dialog2flow/data/final_annotated_dataset.json"
processed_path = "/home/pushpendras0026/dialog2flow/data/processed_transcripts.json"

with open(annotated_path, "r") as f:
    annotated_data = json.load(f)

with open(processed_path, "r") as f:
    processed_data = json.load(f)

# Convert to DataFrames
df_annotated = pd.DataFrame(annotated_data)
df_processed = pd.DataFrame(processed_data)

# Merge on transcript_id, keeping annotated rows
df_final = df_annotated.merge(
    df_processed[['transcript_id', 'satisfaction_score', 'satisfaction_label']],
    on="transcript_id",
    how="left"
)

print("Final shape:", df_final.shape)
print(df_final.head())

output_path = "/home/pushpendras0026/dialog2flow/data/final_annotated_dataset_with_satisfaction.json"
df_final.to_json(output_path, orient="records", indent=2)
print(f"Saved enriched dataset to {output_path}")

Final shape: (18, 8)
                          transcript_id  time_of_interaction   domain  \
0  2ed83b15-d821-4b2c-a5aa-600233f3b72c  2025-11-07 15:54:00   Flight   
1  6b4c8941-cbec-4f30-8f9c-9174f9b6970d  2025-11-08 12:52:00    Hotel   
2  fdee27a2-4679-4474-8691-2390532fff2b  2025-11-10 09:24:00    Hotel   
3  5abd5a28-428c-465c-84a3-8fc02ec37cf3  2025-11-09 11:30:00   Flight   
4  4e7a9492-295b-4de0-97af-7663c31fdba4  2025-11-08 09:22:00  Banking   

                   intent                                    reason_for_call  \
0       Price Sensitivity  Customer called to discuss ticket pricing conc...   
1      Service Complaints  Emma Kim called to report dissatisfaction with...   
2  Discounts & Promotions  Addison called to inquire about a promotional ...   
3        Delay Management  Abigail needed assistance with rebooking her f...   
4            Fraud Alerts  The customer reported a suspected fraudulent t...   

                                        conversation  satis

# Top 20 Conversation Search Pipeline

This notebook demonstrates how to find the top 20 most relevant conversations based on semantic similarity.

## Features:
- Domain-based filtering
- Semantic search using sentence transformers
- Efficient similarity search with Faiss
- Turn-level and transcript-level aggregation

In [ ]:
# Import the pipeline class
import sys
sys.path.append('/home/pushpendras0026/dialog2flow')

from find_top_20_conversations import ConversationSearchPipeline

# Initialize the pipeline
pipeline = ConversationSearchPipeline(model_name='sentence-transformers/all-mpnet-base-v2')

## Example 1: Find Top 20 Banking Conversations

In [ ]:
# Configure your search
DATA_PATH = '/home/pushpendras0026/dialog2flow/data/final_json_for_d2f.json'
DOMAIN = 'Banking'  # Options: Flight, Hotel, Banking, Retail, Telecom, Insurance
QUERY = 'customer frustrated with account fees and overdraft charges'
OUTPUT_DIR = '/home/pushpendras0026/dialog2flow/data/top_20'
TOP_K = 20

# Run the pipeline
results = pipeline.run_pipeline(
    json_file_path=DATA_PATH,
    domain=DOMAIN,
    query=QUERY,
    output_dir=OUTPUT_DIR,
    top_k=TOP_K
)

## Example 2: Inspect Results

In [ ]:
# View the top result details
if results and len(results) > 0:
    top_result = results[0]
    print(f"Transcript ID: {top_result['transcript_id']}")
    print(f"Domain: {top_result['domain']}")
    print(f"Intent: {top_result['intent']}")
    print(f"Reason for Call: {top_result['reason_for_call']}\n")
    
    # Similarity metadata
    sim_meta = top_result['similarity_metadata']
    print(f"Max Similarity Score: {sim_meta['max_similarity_score']:.4f}")
    print(f"Avg Similarity Score: {sim_meta['avg_similarity_score']:.4f}")
    print(f"Matching Turns: {sim_meta['num_matching_turns']}\n")
    
    # Top matching turns
    print("Top 3 Matching Turns:")
    for i, turn in enumerate(sim_meta['top_matching_turns'][:3], 1):
        print(f"\n{i}. {turn['speaker']}: {turn['text']}")
        print(f"   Similarity: {turn['similarity_score']:.4f}")

## Available Domains

Based on the current dataset:
- **Flight**: 3 transcripts
- **Hotel**: 3 transcripts  
- **Banking**: 3 transcripts
- **Retail**: 3 transcripts
- **Telecom**: 3 transcripts
- **Insurance**: 3 transcripts

Note: The current dataset has only 18 transcripts. When you add the full 19k dataset with ~3k per domain, the pipeline will automatically handle the larger volume.